In [1]:
tabla='cbcad10'
dim='dim_casodiag'

In [2]:
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD_2"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [3]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection2)

connection2.close()

In [4]:
base2

,casodiagcod,casodiagnom,casodiagdescor
0,1,NUEVO,N
1,2,REPETIDO,R


In [5]:
base2.columns

Index(['casodiagcod', 'casodiagnom', 'casodiagdescor'], dtype='object')

In [6]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


query_count_before = f"SELECT COUNT(*) FROM {tabla}"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")


#connection3.execute('CREATE TEMPORARY TABLE tmp_cbtci10 ()')
base2.to_sql(name=f'tmp_{tabla}', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cbtci10 FALSO CON LO OBTENIDO DEL ORACLE

query=f"""

ALTER TABLE tmp_{tabla}
ALTER COLUMN casodiagcod  TYPE character(2),
ALTER COLUMN casodiagnom  TYPE character(50);

UPDATE {tabla}
SET 
casodiagcod = t.casodiagcod,
casodiagnom = t.casodiagnom

FROM tmp_{tabla} t 
WHERE {tabla}.casodiagcod = t.casodiagcod AND TRIM({tabla}.casodiagcod) <> '' AND {tabla}.casodiagcod IS NOT NULL;

INSERT INTO {tabla}
(casodiagcod,casodiagnom) 
SELECT 
casodiagcod,casodiagnom

FROM tmp_{tabla}  t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {tabla} 
    WHERE {tabla}.casodiagcod = t.casodiagcod and {tabla}.casodiagcod IS NOT NULL
) ;


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {tabla};
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")
base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection3)


connection3.close()


Cantidad de filas en la tabla cbcad10 antes de la inserción: 2
Cantidad de filas en la tabla cbcad10 después de la inserción: 2
La cantidad de filas insertadas fue de: 0


In [7]:
#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)

#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS
query_count_before = f"SELECT COUNT(*) FROM {dim}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} antes de la inserción: {cant_antes}")

query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN casodiagcod  TYPE character(2),
ALTER COLUMN casodiagnom  TYPE character(50);

INSERT INTO {dim} 

(cod_casodiag,des_casodiag) 

SELECT 

casodiagcod,casodiagnom

FROM tmp_{tabla} t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} d 
    WHERE 
    
    d.cod_casodiag = t.casodiagcod AND
    d.des_casodiag = t.casodiagnom 
);
"""



c1= text(query)
cursor=connection4.execute(c1)

#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {dim};
"""

c2= text(query2)
cursor=connection4.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

ProgrammingError: (psycopg2.errors.UndefinedTable) no existe la relación «dim_casodiag»
LINE 1: SELECT COUNT(*) FROM dim_casodiag
                             ^

[SQL: SELECT COUNT(*) FROM dim_casodiag]
(Background on this error at: https://sqlalche.me/e/14/f405)